# Ch 14 · Lab 3 — 과적합 시각화

원본: `10_Wine_Overfit_Graph.py`

*일부러 데이터를 15%로 줄여서* 과적합이 잘 보이게 하고, train accuracy 와 val_loss 를 한 그래프에 그린다.

## 0. 환경

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from keras import Input, Sequential
from keras.layers import Dense

keras.utils.set_random_seed(0)

print("Keras:", keras.__version__)

from keras.callbacks import ModelCheckpoint, EarlyStopping

## 1. 데이터 — 15% 샘플링

In [ ]:
DATA = "../../data/wine.csv"
df_full = pd.read_csv(DATA, header=None)
df = df_full.sample(frac=0.15, random_state=0).reset_index(drop=True)  # 일부러 작게
X = df.iloc[:, 0:12].to_numpy(dtype="float32")
y = df.iloc[:, 12].to_numpy(dtype="float32")
print("샘플링 후 X:", X.shape)

## 2. 모델 + 긴 학습 (epochs=3500)

In [ ]:
def build_model():
    return Sequential([
        Input(shape=(12,)),
        Dense(30, activation="relu"),
        Dense(12, activation="relu"),
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),
    ])

keras.utils.set_random_seed(0)
model = build_model()
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [ ]:
hist = model.fit(X, y, validation_split=0.33, epochs=3500, batch_size=500, verbose=0)
print("학습 완료")

## 3. 책 스타일 — val_loss(빨강) + acc(파랑) 점도표

In [ ]:
y_vloss = hist.history["val_loss"]
y_acc   = hist.history["accuracy"]   # ← Keras 3: 'acc' 아님

x_len = np.arange(len(y_acc))
plt.figure(figsize=(9, 4))
plt.plot(x_len, y_vloss, "o", c="red",  markersize=2, label="val_loss")
plt.plot(x_len, y_acc,   "o", c="blue", markersize=2, label="train accuracy")
plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Wine 15% — train acc(파랑) ↑ 와 val_loss(빨강) ↑ 가 동시에")
plt.show()

**해석:** 학습이 진행될수록 train accuracy 는 1에 가까워지지만, val_loss 도 **올라감** = 모델이 학습 데이터에 *암기*하기 시작 = 과적합. → Lab 4 의 EarlyStopping 으로 해결.